# Setup

In [ ]:
import json
import os
import numpy as np
import pandas as pd

import floodlight.io.kinexon as knx
import floodlight.io.dfl as dfl

from floodlight.models.kinematics import DistanceModel, VelocityModel
from floodlight.models.kinetics import MetabolicPowerModel
from floodlight.models.geometry import CentroidModel

from floodlight.models.space import DiscreteVoronoiModel

In [ ]:
# Yannik
path_knx = "C:\\Users\\wf4262\\sciebo - Paul, Yannik (wf4262@dshs-koeln.de)@dshs-koeln.sciebo.de\\WMA\\SportVid\\tests"
# path_knx = "D:\DSHS\WMA\SportVid\Sample_Video"

In [ ]:
# Max
positions_filepath = "/home/max/drive/projects/1_SportVid/data/DFL_04_03_positions_raw_observed_DFL-COM-000001_DFL-MAT-J03WMX.xml" 
metadata_filepath = "/home/max/drive/projects/1_SportVid/data/DFL_02_01_matchinformation_DFL-COM-000001_DFL-MAT-J03WMX.xml"      
path_knx = "/home/max/drive/projects/1_SportVid/AP1_DataGeneration/trackingDataSampleVideo.csv"                            

# Unified KPI Data Format

Compute per-frame, per-player KPIs for both DFL and Kinexon formats.

Output structure:
```
{
  frame_idx: [
    [player_id, distance_covered, velocity, metabolic_power],
    ...  # one entry per tracked player
  ],
  ...
}
```

`meta_data` contains:
- `format`: "dfl" or "kinexon"
- `kpi_names`: ["distance_covered", "velocity", "metabolic_power"]
- `player_ids`: {combined_idx → player identifier string}
- `teams`: {team_name → [combined_idx, ...]}
- `segments`: [{"name", "start_frame", "n_frames"}, ...]  (one per half for DFL)
- `framerate`: recording framerate

# DFL

In [ ]:
xy, possession, ballstatus, teamsheet, pitch = dfl.read_position_data_xml(positions_filepath, metadata_filepath)                 
                                                                                                          
for key in xy:                                                                                        
    print(key)  # 'firstHalf', 'secondHalf'                                                             
    for team_name, xy_obj in xy[key].items():                                                         
      print(f'{team_name}: {xy_obj.xy.shape}')  # 'Home', 'Away', 'Ball'                              

sampled_data = xy['firstHalf']['Home'].xy                                                             
print(sampled_data)   

firstHalf
Home: (70708, 40)
Away: (70708, 40)
Ball: (70708, 2)
secondHalf
Home: (75259, 40)
Away: (75259, 40)
Ball: (75259, 2)
[[ 6.900e+00  5.120e+00        nan ...  5.180e+00        nan        nan]
 [ 6.820e+00  5.100e+00        nan ...  5.230e+00        nan        nan]
 [ 6.720e+00  5.080e+00        nan ...  5.280e+00        nan        nan]
 ...
 [-1.000e-02 -1.139e+01        nan ... -1.003e+01        nan        nan]
 [ 0.000e+00 -1.147e+01        nan ... -1.010e+01        nan        nan]
 [ 0.000e+00 -1.154e+01        nan ... -1.018e+01        nan        nan]]


In [58]:
DistMod = DistanceModel()
VelMod = VelocityModel()
MetPowMod = MetabolicPowerModel()
CentMod = CentroidModel()

kpi_dict = {}
for half in ["firstHalf", "secondHalf"]:
    for team in ["Home", "Away"]:
        print(f"{half} - {team}")
        DistMod.fit(xy[half][team])
        VelMod.fit(xy[half][team])
        MetPowMod.fit(xy[half][team])
        CentMod.fit(xy[half][team])

        kpi_dict[f'distance_covered_{half}_{team}'] = np.array(DistMod.cumulative_distance_covered())
        kpi_dict[f'max_velocity_{half}_{team}'] = np.nanmax(VelMod.velocity(), axis=0).round(2)
        kpi_dict[f'metabolic_power_{half}_{team}'] = np.array(MetPowMod.metabolic_power())
        kpi_dict[f'centroid_{half}_{team}'] = CentMod.centroid().xy


firstHalf - Home


/tmp/ipykernel_270336/2027585810.py:16: RuntimeWarning: All-NaN axis encountered
  kpi_dict[f'max_velocity_{half}_{team}'] = np.nanmax(VelMod.velocity(), axis=0).round(2)


firstHalf - Away


/tmp/ipykernel_270336/2027585810.py:16: RuntimeWarning: All-NaN axis encountered
  kpi_dict[f'max_velocity_{half}_{team}'] = np.nanmax(VelMod.velocity(), axis=0).round(2)


secondHalf - Home


/tmp/ipykernel_270336/2027585810.py:16: RuntimeWarning: All-NaN axis encountered
  kpi_dict[f'max_velocity_{half}_{team}'] = np.nanmax(VelMod.velocity(), axis=0).round(2)


secondHalf - Away


/tmp/ipykernel_270336/2027585810.py:16: RuntimeWarning: All-NaN axis encountered
  kpi_dict[f'max_velocity_{half}_{team}'] = np.nanmax(VelMod.velocity(), axis=0).round(2)


# Kinexon

In [ ]:
# Load data
csv = pd.read_csv(os.path.join(path_knx, "sportvid_newphase_10sec_centered_30Hz.csv"), delimiter=";")
knx_teamsheets = knx.read_teamsheets_from_csv(os.path.join(path_knx, "sportvid_newphase_10sec_centered_30Hz.csv"), delimiter=";")
knx_pos = knx.read_position_data_csv(os.path.join(path_knx, "sportvid_newphase_10sec_centered_30Hz.csv"), delimiter=";")

# csv = pd.read_csv(os.path.join(path_knx, "trackingDataSampleVideo.csv"), delimiter=",")
# knx_teamsheets = knx.read_teamsheets_from_csv(os.path.join(path_knx, "trackingDataSampleVideo.csv"), delimiter=",")
# knx_pos = knx.read_position_data_csv(os.path.join(path_knx, "trackingDataSampleVideo.csv"), delimiter=",")

In [5]:
group_id_map = csv[["number", "group id"]].drop_duplicates(subset=["number"])
for idx, i in enumerate(knx_teamsheets):
    knx_teamsheets[idx].teamsheet = pd.merge(
        i.teamsheet.astype({"number": int}),
        group_id_map,
        on=["number"]
    )

In [6]:
# Hardcoded pos_meta matching the frontend metaDataTopView format
pos_meta = {
    "player_ids": {
        "1":  {"id": 29, "name": " Player 29", "number": 29},
        "2":  {"id": 95, "name": " Ball 5",    "number": 95},
        "3":  {"id": 9,  "name": " Player 9",  "number": 9},
        "4":  {"id": 5,  "name": " Player 5",  "number": 5},
        "5":  {"id": 15, "name": " Player 15", "number": 15},
        "6":  {"id": 17, "name": " Player 17", "number": 17},
        "7":  {"id": 19, "name": " Player 19", "number": 19},
        "8":  {"id": 8,  "name": " Player 8",  "number": 8},
        "9":  {"id": 25, "name": " Player 25", "number": 25},
        "10": {"id": 18, "name": " Player 18", "number": 18},
        "11": {"id": 10, "name": " Player 10", "number": 10},
        "12": {"id": 2,  "name": " Player 2",  "number": 2},
        "13": {"id": 12, "name": " Player 12", "number": 12},
        "14": {"id": 21, "name": " Player 21", "number": 21},
        "15": {"id": 23, "name": " Player 23", "number": 23},
        "16": {"id": 16, "name": " Player 16", "number": 16},
        "17": {"id": 1,  "name": " Player 1",  "number": 1},
        "18": {"id": 11, "name": " Player 11", "number": 11},
        "19": {"id": 20, "name": " Player 20", "number": 20},
        "20": {"id": 6,  "name": " Player 6",  "number": 6},
        "21": {"id": 26, "name": " Player 26", "number": 26},
        "22": {"id": 7,  "name": " Player 7",  "number": 7},
        "23": {"id": 22, "name": " Player 22", "number": 22},
        "24": {"id": 14, "name": " Player 14", "number": 14},
        "25": {"id": 4,  "name": " Player 4",  "number": 4},
        "26": {"id": 3,  "name": " Player 3",  "number": 3},
        "27": {"id": 24, "name": " Player 24", "number": 24},
        "28": {"id": 97, "name": " Ball 3",    "number": 97},
        "29": {"id": 13, "name": " Player 13", "number": 13},
        "30": {"id": 27, "name": " Player 27", "number": 27},
    },
    "team_ids": {
        "1": {"id": 5, "name": "ball"},
        "2": {"id": 3, "name": "A"},
        "3": {"id": 1, "name": "A"},
        "4": {"id": 2, "name": "B"},
    },
}

# Build reverse lookups (same logic as kpi_computation.py)
team_id_by_orig   = {}  # str(original_id) → pos_meta int_id (key)
player_id_by_orig = {}  # str(original_id) → pos_meta int_id (key)
ball_group_id_str = None

for int_id_str, info in pos_meta["team_ids"].items():
    team_id_by_orig[info["id"]] = int(int_id_str)

for int_id_str, info in pos_meta["player_ids"].items():
    player_id_by_orig[info["id"]] = int(int_id_str)

ball_entry = pos_meta.get("team_ids", {}).get("1")
if ball_entry:
    ball_group_id_str = str(ball_entry["id"])

print("team_id_by_orig:", team_id_by_orig)
print("player_id_by_orig:", player_id_by_orig)
print("ball_group_id_str:", ball_group_id_str)

team_id_by_orig: {5: 1, 3: 2, 1: 3, 2: 4}
player_id_by_orig: {29: 1, 95: 2, 9: 3, 5: 4, 15: 5, 17: 6, 19: 7, 8: 8, 25: 9, 18: 10, 10: 11, 2: 12, 12: 13, 21: 14, 23: 15, 16: 16, 1: 17, 11: 18, 20: 19, 6: 20, 26: 21, 7: 22, 22: 23, 14: 24, 4: 25, 3: 26, 24: 27, 97: 28, 13: 29, 27: 30}
ball_group_id_str: 5


In [ ]:
ball_team_ids = set()  # group id values that are the ball

for idx, ts in enumerate(knx_teamsheets):
    df = ts.teamsheet.copy()
    group_id_val = df['group id'].iloc[0]
    tid = team_id_by_orig.get(group_id_val, group_id_val)

    if tid == 1:  # posdata_convert always maps ball → team id 1
        ball_team_ids.add(group_id_val)
        continue

    df['pid'] = df['number'].map(player_id_by_orig)  # jersey number → pos_meta player int_id
    df['tid'] = tid                                   # pos_meta team int_id (same for all in group)
    knx_teamsheets[idx].teamsheet = df

framerate = int(knx_pos[0].framerate) if knx_pos and knx_pos[0].framerate else 25
print("Ball group ids:", ball_team_ids)
print("Framerate:", framerate)
print(knx_teamsheets[0].teamsheet[['number', "pid","group id","tid"]].head())

tid: 3 group_id_val: 1
tid: 2 group_id_val: 3
tid: 1 group_id_val: 5
Ball group ids: {np.int64(5)}
Framerate: 30
   number  pid  group id  tid
0       9    3         1    3
1       5    4         3    3
2      15    5         3    3
3       8    8         1    3
4      10   11         3    3


In [8]:
all_frame_kpis = {}  # {frame_idx: [[pid, tid, dist, vel, metpow], ...]}

team_kpi_arrays = {}  # i → (df_ts_sorted, dist_arr, vel_arr, metpow_arr)
n_frames = None

for i, xy_obj in enumerate(knx_pos):
    df_ts = knx_teamsheets[i].teamsheet
    if df_ts['group id'].iloc[0] in ball_team_ids:
        continue

    dist_mod = DistanceModel()
    vel_mod = VelocityModel()
    metpow_mod = MetabolicPowerModel()
    dist_mod.fit(xy_obj)
    vel_mod.fit(xy_obj)
    metpow_mod.fit(xy_obj)

    dist_arr = np.array(dist_mod.cumulative_distance_covered()).round(2)  # (T, N)
    vel_arr  = np.array(vel_mod.velocity()).round(2)                       # (T, N)
    metpow_arr = np.array(metpow_mod.metabolic_power()).round(2)          # (T, N)

    team_kpi_arrays[i] = (df_ts.sort_values("xID").reset_index(drop=True), dist_arr, vel_arr, metpow_arr)
    if n_frames is None:
        n_frames = dist_arr.shape[0]

if n_frames is not None:
    for i, (df_sorted, dist_arr, vel_arr, metpow_arr) in team_kpi_arrays.items():
        n_players = dist_arr.shape[1]
        dist_list = dist_arr.tolist()
        vel_list  = vel_arr.tolist()
        metpow_list = metpow_arr.tolist()

        for frame_idx in range(n_frames):
            if frame_idx not in all_frame_kpis:
                all_frame_kpis[frame_idx] = []
            for p in range(n_players):
                row = df_sorted.iloc[p] if p < len(df_sorted) else None
                pid = int(row['pid']) if row is not None and pd.notna(row['pid']) else -1
                tid = int(row['tid']) if row is not None else -1
                d = dist_list[frame_idx][p]
                v = vel_list[frame_idx][p]
                m = metpow_list[frame_idx][p]
                all_frame_kpis[frame_idx].append([
                    pid, tid,
                    None if d != d else d,
                    None if v != v else v,
                    None if m != m else m,
                ])

print(f"Computed KPIs for {n_frames} frames, {sum(len(v) for v in all_frame_kpis.values())} total player-frame entries")
print("Sample frame 0:", all_frame_kpis.get(0))

Computed KPIs for 300 frames, 8400 total player-frame entries
Sample frame 0: [[3, 3, 0.02, 0.74, 1.71], [4, 3, 0.0, 0.07, 0.29], [5, 3, 0.01, 0.42, 0.86], [8, 3, 0.2, 6.13, 14.65], [11, 3, 0.0, 0.04, 0.14], [12, 3, 0.19, 5.82, 12.44], [13, 3, 0.09, 2.8, 4.94], [17, 3, 0.24, 7.31, 22.63], [18, 3, 0.18, 5.33, 12.7], [20, 3, 0.15, 4.4, 8.12], [22, 3, 0.13, 3.78, 7.13], [24, 3, 0.13, 3.88, 7.33], [25, 3, 0.1, 3.01, 5.6], [26, 3, 0.14, 4.2, 7.93], [29, 3, 0.0, None, None], [1, 2, 0.05, 1.5, 2.5], [6, 2, 0.18, 5.33, 11.71], [7, 2, 0.11, 3.18, 6.01], [9, 2, 0.17, 5.21, 11.9], [10, 2, 0.11, 3.38, 6.11], [14, 2, 0.07, 2.04, 3.83], [15, 2, 0.25, 7.55, 17.06], [16, 2, 0.23, 6.95, 18.08], [19, 2, 0.27, 8.05, 26.46], [21, 2, 0.14, 4.33, 8.37], [23, 2, 0.14, 4.08, 8.52], [27, 2, 0.13, 3.78, 6.68], [30, 2, 0.0, None, None]]


In [12]:
# Frame index → milliseconds
freq = 1000.0 / framerate
all_frame_kpis_ms = {
    int(np.rint(frame_idx * freq)): players
    for frame_idx, players in all_frame_kpis.items()
}

meta = {
    "format": "kinexon",
    "kpi_names": ["distance_covered", "velocity", "metabolic_power"],
    "player_ids": pos_meta["player_ids"],
    "team_ids": pos_meta["team_ids"],
    "framerate": framerate,
}

print("Meta:", json.dumps(meta, indent=2))
print(f"\nTotal timestamps: {len(all_frame_kpis_ms)}")
print("Sample timestamp 0ms:", all_frame_kpis_ms.get(0))

Meta: {
  "format": "kinexon",
  "kpi_names": [
    "distance_covered",
    "velocity",
    "metabolic_power"
  ],
  "player_ids": {
    "1": {
      "id": 29,
      "name": " Player 29",
      "number": 29
    },
    "2": {
      "id": 95,
      "name": " Ball 5",
      "number": 95
    },
    "3": {
      "id": 9,
      "name": " Player 9",
      "number": 9
    },
    "4": {
      "id": 5,
      "name": " Player 5",
      "number": 5
    },
    "5": {
      "id": 15,
      "name": " Player 15",
      "number": 15
    },
    "6": {
      "id": 17,
      "name": " Player 17",
      "number": 17
    },
    "7": {
      "id": 19,
      "name": " Player 19",
      "number": 19
    },
    "8": {
      "id": 8,
      "name": " Player 8",
      "number": 8
    },
    "9": {
      "id": 25,
      "name": " Player 25",
      "number": 25
    },
    "10": {
      "id": 18,
      "name": " Player 18",
      "number": 18
    },
    "11": {
      "id": 10,
      "name": " Player 10",
      "numbe